# Phantom arguments: what does a model do when the tool cannot do what was asked?

Each item shows the model ONE tool definition (JSON Schema, `additionalProperties: false`) and ONE user
request. The model answers with one JSON tool call written as text. The request asks for something the tool
cannot do. Scoring is JSON Schema validation plus fixed per-item checks. There is no LLM judge.

- Items v1.1: 202 items, 75 invented tools, eight families.
  v1.0 (162 items, unchanged): `conventional`, `near_miss`, `inexpressible`, `enum_pressure`, `nested`, `control`.
  v1.1 adds `repurposing` (28: the tool has a tempting parameter that means something else, e.g. `cc` for a
  BCC request) and `matched_control` (12: the same parameter is the right one).
- Three prompt conditions, one task each: `neutral` (the model must call), `instructed` (one extra sentence:
  use only the schema's parameters), `may_decline` (one extra sentence: it may answer
  `{"cannot_do": "..."}` instead of calling).
- Tool calls are rendered as plain text. This is NOT a provider's native tool-calling API.

**Generated file** (kaggle/build_notebook.py in the project repo). Built with AI assistance.

## What is verified and what is not
The prompt builder, parser, scorer, report and items are the code that produced the local runs and are
covered by the project's tests. Every call into `kaggle_benchmarks` lives in ONE cell, "Kaggle adapter",
except the main task's `@kbench.task(...)` decorator, which the editor's task panel needs to see literally.
This notebook was run end to end on Kaggle on 2026-10-02 (kaggle-benchmarks 0.6.1, model
google/gemini-3.7-flash). If something fails on Kaggle later, the adapter cell is the place to patch.

Editor rules learned the hard way: `TASK_NAME` must be a plain string literal, the main task's decorator must
be a literal, top-level `@kbench.task(name="...")` (the editor panel and `kaggle b t push` both read the name
without running code), and an HTML/SVG tag name in angle brackets anywhere in the source (the SVG tag "text",
in a docstring) hides every task from the panel.

Command line (Kaggle CLI 2.2.4): `kaggle b t push phantom-args-neutral -f benchmark_notebook.py --wait`, then
`kaggle b t run phantom-args-neutral -m MODEL`. Each run executes this file on Kaggle against one model;
`kaggle b t download` returns the run's files, including phantom_records.jsonl (every reply, its cost and usage).

In [ ]:
# ----------------------------------------------------------------------------- configuration
CONDITION = "instructed"         # "neutral" (must call), "instructed", or "may_decline". One task per condition.
SUBSET = "full"               # "full": all 202 items (the leaderboard setting: every model gets the same items).
                              # "core": 10 items per family (80).
                              # "auto": every item if the probe projects <= PER_MODEL_BUDGET_USD, else the core subset
ONLY_FAMILIES = None          # None = all eight families, or a list such as ["repurposing", "matched_control"]
PER_MODEL_BUDGET_USD = 0.60   # used by SUBSET = "auto"
SESSION_BUDGET_USD = 2.00     # no paid call is started once this run has spent this much (daily quota: $10;
                              # a full run cost at most about $1.20 per model and condition on 2026-10-02)
MAX_CALLS_IF_COST_UNKNOWN = 400   # fallback guard when the library reports no cost for a call
EXTRA_MODELS = []             # model slugs to ALSO run in this session, e.g. ["google/gemini-2.5-flash"]; [] = only the task's model
PARALLEL_JOBS = 8             # 0 = plain sequential loop (smallest API surface, slowest)
TEMPERATURE = 0
MAX_OUTPUT_TOKENS = 8000      # sent as max_tokens. Kaggle's Model Proxy reserves quota for every call from its output
                              # cap and refuses calls (403) once reservations exceed the quota left; without a cap it
                              # reserves the model's full maximum. Largest reply seen on Kaggle: 2,144 tokens (local
                              # runs used 3,000). A reply that reaches the cap is kept, scored, and marked truncated.
CACHE_DIR = "/kaggle/working"  # replies are appended here as they arrive; a re-run never repeats a cached call
TASK_NAME = "phantom_args_instructed" # a plain literal on purpose: the editor's Benchmark Task panel reads task names
                                     # without running code and shows an f-string unresolved. Change it together with
                                     # CONDITION, the main task's decorator and the %choose line in the last cell.
assert TASK_NAME == f"phantom_args_{CONDITION}", "set TASK_NAME (and the %choose line) to phantom_args_<CONDITION>"

In [ ]:
import base64
import glob
import json
import math
import os
import threading
import time
import zlib
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Iterable, Iterator

if not os.path.isdir(CACHE_DIR):          # running outside Kaggle
    CACHE_DIR = "."

In [ ]:
# ----------------------------------------------------------------------------- prompt builder
# Inlined verbatim from phantom_args/prompt.py (project repo); covered by the project's tests.
"""Prompt builder.

The whole item is rendered into ONE user message so that the exact same text
can be sent through any chat API (local runner) and through
`llm.prompt(...)` in a Kaggle notebook. No native tool-calling is used.

Three conditions:
  neutral     - the base prompt: the model must call
  instructed  - the base prompt plus one sentence (use only the schema's parameters)
  may_decline - the base prompt plus one sentence that allows a decline in a
                fixed JSON form instead of a call (added in v1.1)
"""

import hashlib
import json

CONDITIONS = ("neutral", "instructed", "may_decline")
# The two conditions of v1.0. `may_decline` was added in v1.1 and changes no v1.0 prompt.
BASE_CONDITIONS = ("neutral", "instructed")

# The one sentence that separates the two conditions.
INSTRUCTION_SENTENCE = (
    "Use only the parameters defined in the tool's schema; do not add any "
    "parameter that is not listed there."
)

# The one sentence of the third condition: a way out, in a fixed format so that
# a decline can be read without any judgement.
DECLINE_KEY = "cannot_do"
DECLINE_SENTENCE = (
    "If the tool cannot do what the user asked, do not call it: reply instead "
    'with exactly this JSON object and nothing else: {"cannot_do": "<one '
    'sentence saying what the tool cannot do>"}.'
)

_HEADER = (
    "You are an assistant that completes user requests by calling tools. "
    "You will be shown the available tool definition(s) and one user request. "
    "Reply with exactly one tool call and nothing else, as a single JSON object "
    'of the form {"name": "<tool name>", "arguments": {...}}. '
    "After your call you will see the tool's result and can then answer the user."
)


def render_tool(tool: dict) -> str:
    """Compact one-line JSON, the way most chat templates serialise tools."""
    return json.dumps(
        {
            "name": tool["name"],
            "description": tool["description"],
            "parameters": tool["parameters"],
        },
        ensure_ascii=False,
        separators=(", ", ": "),
    )


def build_prompt(item: dict, condition: str = "neutral") -> str:
    if condition not in CONDITIONS:
        raise ValueError(f"unknown condition {condition!r}")
    tools = item["tools"]
    header = _HEADER
    if condition == "instructed":
        header = header + " " + INSTRUCTION_SENTENCE
    elif condition == "may_decline":
        header = header + " " + DECLINE_SENTENCE
    label = "Available tool:" if len(tools) == 1 else "Available tools:"
    rendered = "\n".join(render_tool(t) for t in tools)
    return f"{header}\n\n{label}\n{rendered}\n\nUser request: {item['request']}"


def build_messages(item: dict, condition: str = "neutral") -> list[dict]:
    return [{"role": "user", "content": build_prompt(item, condition)}]


def prompt_sha(item: dict, condition: str = "neutral") -> str:
    """Short hash of the exact prompt text; stored with every reply so that a
    reply recorded for an older version of an item is never scored against the
    current one."""
    return hashlib.sha1(build_prompt(item, condition).encode("utf-8")).hexdigest()[:12]

In [ ]:
# ----------------------------------------------------------------------------- tolerant parser
# Inlined verbatim from phantom_args/parser.py (project repo); covered by the project's tests.
"""Tolerant tool-call parser.

Turns a model's free-text reply into `ParsedCall(name, arguments)` or a parse
failure with a reason. It never guesses about schema validity; that is the
scorer's job.

Tolerated:
  - code fences, <tool_call> tags, prose before/after the JSON
  - leaked <think>...</think> blocks
  - trailing commas, Python-literal dicts (single quotes, True/False/None)
  - `arguments` given as a JSON string
  - wrapper shapes: {"tool_call": {...}}, {"function": {...}}, {"tool_calls": [...]}
  - alias keys: tool / tool_name / function for the name;
    args / parameters / input / params for the arguments
  - a "flattened" call {"name": "t", "query": ...} and a bare arguments object
    (both flagged as format deviations)

Not tolerated (reported as their own outcome, never as a phantom argument):
  - empty reply                                -> reason "empty"
  - no JSON object at all                      -> reason "no_json"
  - JSON that is not shaped like a tool call   -> reason "not_a_tool_call"
  - two or more *different* tool calls         -> reason "multiple_calls"
  - braces present but no complete call decodes -> reason "malformed_json"
    (prose with no "{" at all is "no_json": the model answered in words)
  - arguments that are not a JSON object       -> reason "arguments_not_object"

Declines (v1.1, `may_decline` condition only, `allow_decline=True`):
  - {"cannot_do": "TEXT"} and no call         -> declined, reason "declined"
  - a decline together with a named call       -> reason "call_and_decline"
  A refusal written as prose is still "no_json": the format was not followed.

The tool library never uses a top-level parameter called name / tool /
function / arguments / args / parameters / params / input / type / id (the
item builder enforces this), so the "flattened" shape is unambiguous.
"""

import ast
import json
import re
from dataclasses import dataclass, field
from typing import Any

DECLINE_KEY = "cannot_do"
NAME_KEYS = ("name", "tool", "tool_name", "function_name", "function")
ARG_KEYS = ("arguments", "args", "parameters", "params", "input", "tool_input")

_THINK_RE = re.compile(r"<think(?:ing)?>.*?</think(?:ing)?>", re.DOTALL | re.IGNORECASE)
_TRAILING_COMMA_RE = re.compile(r",(\s*[}\]])")


@dataclass
class ParsedCall:
    ok: bool
    name: str | None = None
    arguments: Any = None
    reason: str | None = None          # set when ok is False
    notes: list[str] = field(default_factory=list)   # format deviations that were tolerated
    n_candidates: int = 0
    declined: bool = False             # may_decline condition only: the reply was {"cannot_do": "..."}
    decline_text: str | None = None

    def to_dict(self) -> dict:
        return {
            "ok": self.ok,
            "name": self.name,
            "arguments": self.arguments,
            "reason": self.reason,
            "notes": list(self.notes),
        }


# --------------------------------------------------------------------------
# low-level helpers
# --------------------------------------------------------------------------
def strip_think(text: str) -> str:
    """Remove leaked reasoning blocks; also handles a dangling `</think>`."""
    text = _THINK_RE.sub("", text)
    low = text.lower()
    for closer in ("</think>", "</thinking>"):
        idx = low.rfind(closer)
        if idx != -1:
            text = text[idx + len(closer):]
            low = text.lower()
    return text


def _balanced_objects(text: str) -> list[str]:
    """Return every outermost {...} substring, honouring string literals."""
    out: list[str] = []
    depth = 0
    start = -1
    quote: str | None = None
    escape = False
    for i, ch in enumerate(text):
        if quote is not None:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == quote:
                quote = None
            continue
        if depth > 0 and ch in ('"', "'"):
            # Only treat ' as a string delimiter when it plausibly opens a
            # Python-style string (after one of  { [ , :  ), so that an
            # apostrophe inside prose never swallows the rest of the text.
            if ch == "'":
                j = i - 1
                while j >= 0 and text[j] in " \t\r\n":
                    j -= 1
                if j < 0 or text[j] not in "{[,:":
                    continue
            quote = ch
            continue
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            if depth > 0:
                depth -= 1
                if depth == 0 and start != -1:
                    out.append(text[start:i + 1])
                    start = -1
    return out


def _raw_decode_objects(text: str) -> list[str]:
    """Fallback scan: try a strict JSON decode at every "{".

    Used only when the balanced-brace scan finds nothing decodable, e.g. when
    a stray "{" in the prose before the call unbalances the whole reply.
    """
    decoder = json.JSONDecoder()
    out: list[str] = []
    i = 0
    while True:
        i = text.find("{", i)
        if i == -1:
            return out
        try:
            value, end = decoder.raw_decode(text, i)
        except (json.JSONDecodeError, RecursionError):
            i += 1
            continue
        if isinstance(value, dict):
            out.append(text[i:end])
        i = end


def loads_lenient(blob: str) -> tuple[Any, list[str]]:
    """json.loads with two repairs. Raises ValueError if nothing works."""
    try:
        return json.loads(blob), []
    except (json.JSONDecodeError, RecursionError):
        pass
    repaired = _TRAILING_COMMA_RE.sub(r"\1", blob)
    if repaired != blob:
        try:
            return json.loads(repaired), ["trailing_comma"]
        except (json.JSONDecodeError, RecursionError):
            pass
    try:
        value = ast.literal_eval(repaired)
    except (ValueError, SyntaxError, MemoryError, RecursionError, TypeError):
        raise ValueError("not JSON") from None
    if isinstance(value, (dict, list)):
        return value, ["python_literal"]
    raise ValueError("not JSON")


def _looks_like_tool_definition(obj: dict) -> bool:
    """A model sometimes echoes the tool spec; that is not a call."""
    params = obj.get("parameters")
    return (
        isinstance(params, dict)
        and params.get("type") == "object"
        and "properties" in params
        and ("description" in obj or "required" in params or "additionalProperties" in params)
    )


def _coerce_arguments(value: Any, notes: list[str]) -> Any:
    """Arguments may arrive as a JSON string; decode once (twice if nested)."""
    for _ in range(2):
        if isinstance(value, str):
            stripped = value.strip()
            if stripped.startswith("{"):
                try:
                    value, extra = loads_lenient(stripped)
                except ValueError:
                    return value
                if "arguments_as_string" not in notes:
                    notes.append("arguments_as_string")
                for n in extra:
                    if n not in notes:
                        notes.append(n)
                continue
        break
    return value


def _extract_calls(obj: Any, notes: list[str]) -> list[tuple[str | None, Any, list[str]]]:
    """Normalise one decoded JSON value into zero or more (name, args, notes)."""
    if isinstance(obj, list):
        calls: list = []
        for el in obj:
            calls.extend(_extract_calls(el, notes))
        return calls
    if not isinstance(obj, dict):
        return []
    if _looks_like_tool_definition(obj):
        return []

    # wrappers
    for wrapper in ("tool_call", "function_call"):
        if isinstance(obj.get(wrapper), dict):
            return _extract_calls(obj[wrapper], notes + [f"wrapped:{wrapper}"])
    if isinstance(obj.get("tool_calls"), list):
        return _extract_calls(obj["tool_calls"], notes + ["wrapped:tool_calls"])
    if isinstance(obj.get("function"), dict):
        return _extract_calls(obj["function"], notes + ["wrapped:function"])

    name_key = next((k for k in NAME_KEYS if isinstance(obj.get(k), str)), None)
    arg_key = next((k for k in ARG_KEYS if k in obj), None)
    local = list(notes)

    if name_key and arg_key:
        if name_key != "name":
            local.append(f"name_key:{name_key}")
        if arg_key != "arguments":
            local.append(f"arguments_key:{arg_key}")
        args = _coerce_arguments(obj[arg_key], local)
        return [(obj[name_key], args, local)]

    if name_key and not arg_key:
        rest = {k: v for k, v in obj.items() if k != name_key and k not in ("type", "id")}
        if rest:
            local.append("flattened_arguments")
        else:
            local.append("no_arguments_key")
        if name_key != "name":
            local.append(f"name_key:{name_key}")
        return [(obj[name_key], rest, local)]

    if arg_key and not name_key and len(obj) == 1:
        local.append("missing_name")
        args = _coerce_arguments(obj[arg_key], local)
        return [(None, args, local)]

    # No name at all: treat the whole object as a bare arguments object.
    local.append("bare_arguments")
    return [(None, obj, local)]


# --------------------------------------------------------------------------
# public API
# --------------------------------------------------------------------------
def _is_decline(value: Any) -> bool:
    """{"cannot_do": "..."} and nothing that looks like a call."""
    return (isinstance(value, dict) and DECLINE_KEY in value
            and not any(k in value for k in NAME_KEYS + ARG_KEYS))


def parse_tool_call(text: str | None, known_tools: list[str] | None = None,
                    allow_decline: bool = False) -> ParsedCall:
    """Parse a model reply into exactly one tool call.

    With `allow_decline` (the `may_decline` prompt condition only) a reply
    whose JSON is {"cannot_do": "..."} is returned as a decline
    (`declined=True`, `ok=False`, reason "declined"). A reply that both
    declines and calls is a parse failure ("call_and_decline"). Without the
    flag nothing changes: v1.0 conditions are parsed exactly as before.

    `known_tools` (the tool names shown in the prompt) is only used to
    interpret a reply whose arguments were given without a name wrapper: when
    exactly one tool was offered, the bare arguments are attributed to it.
    """
    if text is None or not str(text).strip():
        return ParsedCall(ok=False, reason="empty")
    cleaned = strip_think(str(text))

    decoded: list[tuple[Any, list[str]]] = []
    for blob in _balanced_objects(cleaned):
        try:
            value, repair_notes = loads_lenient(blob)
        except ValueError:
            continue
        decoded.append((value, repair_notes))
    if not decoded:
        # Nothing balanced decodes. Scan for any decodable object, but accept
        # only complete named calls: a fragment from inside a broken call (one
        # array item, one nested object) must not be mistaken for the call.
        fragments = _raw_decode_objects(cleaned)
        for blob in fragments:
            value = json.loads(blob)
            if (allow_decline and _is_decline(value)) or any(
                    name is not None for name, _, _ in _extract_calls(value, [])):
                decoded.append((value, ["recovered_by_scan"]))
        if not decoded:
            broken = bool(fragments) or "{" in cleaned
            return ParsedCall(ok=False, reason="malformed_json" if broken else "no_json",
                              n_candidates=len(fragments))

    declines: list[Any] = []
    if allow_decline:
        declines = [v for v, _ in decoded if _is_decline(v)]
        decoded = [(v, n) for v, n in decoded if not _is_decline(v)]

    calls: list[tuple[str | None, Any, list[str]]] = []
    for value, repair_notes in decoded:
        calls.extend(_extract_calls(value, list(repair_notes)))
    if declines:
        if any(name is not None for name, _, _ in calls):
            return ParsedCall(ok=False, reason="call_and_decline", n_candidates=len(calls) + len(declines))
        reason_text = declines[0].get(DECLINE_KEY)
        return ParsedCall(ok=False, reason="declined", declined=True,
                          decline_text=reason_text if isinstance(reason_text, str) else json.dumps(reason_text))
    if not calls:
        return ParsedCall(ok=False, reason="not_a_tool_call", n_candidates=len(decoded))

    # Prefer properly named calls over bare-argument guesses when both exist.
    named = [c for c in calls if c[0] is not None]
    if named:
        calls = named

    # De-duplicate identical calls (a model repeating itself is still one call).
    unique: list[tuple[str | None, Any, list[str]]] = []
    seen: set[str] = set()
    for name, args, notes in calls:
        try:
            key = json.dumps([name, args], sort_keys=True, default=str)
        except (TypeError, ValueError):
            key = repr((name, args))
        if key not in seen:
            seen.add(key)
            unique.append((name, args, notes))
    if len(unique) > 1:
        return ParsedCall(ok=False, reason="multiple_calls", n_candidates=len(unique))

    name, args, notes = unique[0]
    if len(calls) > 1 and "repeated_call" not in notes:
        notes.append("repeated_call")
    if name is None:
        if known_tools and len(known_tools) == 1:
            name = known_tools[0]
        else:
            return ParsedCall(ok=False, reason="not_a_tool_call", notes=notes, n_candidates=1)
    if not isinstance(args, dict):
        # arguments must be an object; anything else cannot be validated as a call
        return ParsedCall(ok=False, reason="arguments_not_object", name=name, notes=notes, n_candidates=1)
    return ParsedCall(ok=True, name=name.strip(), arguments=args, notes=notes, n_candidates=1)

In [ ]:
# ----------------------------------------------------------------------------- fallback validator (used only if `jsonschema` is not installed)
# Inlined verbatim from phantom_args/minischema.py (project repo); covered by the project's tests.
"""A tiny JSON Schema validator used only when `jsonschema` is not installed.

It covers exactly the keywords the tool library uses (the item builder rejects
anything else): type, enum, properties, required, additionalProperties: false,
items, minimum, maximum. Errors mimic the attributes of jsonschema's
ValidationError that the scorer reads. tests/test_minischema.py checks that it
agrees with jsonschema on every item, every trap and a set of nasty inputs.
"""

from dataclasses import dataclass, field
from typing import Any, Iterator


@dataclass
class MiniError:
    validator: str
    validator_value: Any
    instance: Any
    schema: dict
    absolute_path: list = field(default_factory=list)
    message: str = ""


def _is_type(value: Any, name: str) -> bool:
    if name == "string":
        return isinstance(value, str)
    if name == "boolean":
        return isinstance(value, bool)
    if name == "integer":
        if isinstance(value, bool):
            return False
        return isinstance(value, int) or (isinstance(value, float) and value.is_integer())
    if name == "number":
        return isinstance(value, (int, float)) and not isinstance(value, bool)
    if name == "object":
        return isinstance(value, dict)
    if name == "array":
        return isinstance(value, list)
    if name == "null":
        return value is None
    raise ValueError(f"unsupported type {name!r}")


def _enum_match(value: Any, options: list) -> bool:
    for opt in options:
        if isinstance(value, bool) != isinstance(opt, bool):
            continue
        if value == opt:
            return True
    return False


def iter_errors(schema: dict, instance: Any, path: list | None = None) -> Iterator[MiniError]:
    path = path or []
    if "type" in schema and not _is_type(instance, schema["type"]):
        yield MiniError("type", schema["type"], instance, schema, list(path),
                        f"{instance!r} is not of type {schema['type']!r}")
    if "enum" in schema and not _enum_match(instance, schema["enum"]):
        yield MiniError("enum", schema["enum"], instance, schema, list(path),
                        f"{instance!r} is not one of {schema['enum']!r}")
    if isinstance(instance, (int, float)) and not isinstance(instance, bool):
        if "minimum" in schema and instance < schema["minimum"]:
            yield MiniError("minimum", schema["minimum"], instance, schema, list(path),
                            f"{instance!r} is less than the minimum of {schema['minimum']!r}")
        if "maximum" in schema and instance > schema["maximum"]:
            yield MiniError("maximum", schema["maximum"], instance, schema, list(path),
                            f"{instance!r} is greater than the maximum of {schema['maximum']!r}")
    if isinstance(instance, dict):
        props = schema.get("properties", {})
        for name in schema.get("required", []):
            if name not in instance:
                yield MiniError("required", schema["required"], instance, schema, list(path),
                                f"{name!r} is a required property")
        if schema.get("additionalProperties") is False:
            extras = [k for k in instance if k not in props]
            if extras:
                yield MiniError("additionalProperties", False, instance, schema, list(path),
                                f"Additional properties are not allowed ({', '.join(map(repr, extras))})")
        for key, sub in props.items():
            if key in instance:
                yield from iter_errors(sub, instance[key], path + [key])
    if isinstance(instance, list) and isinstance(schema.get("items"), dict):
        for i, el in enumerate(instance):
            yield from iter_errors(schema["items"], el, path + [i])


class MiniValidator:
    def __init__(self, schema: dict):
        self.schema = schema

    def iter_errors(self, instance: Any) -> Iterator[MiniError]:
        return iter_errors(self.schema, instance)

In [ ]:
# ----------------------------------------------------------------------------- deterministic scorer
# Inlined verbatim from phantom_args/scorer.py (project repo); covered by the project's tests.
"""Deterministic scorer: JSON Schema validation only, no LLM judge.

For one (item, model reply) pair it produces a `Score` with

  label   - one primary label (see LABEL_ORDER)
  flags   - every fault that was found (an answer can have several)
  phantom_keys - the unknown argument names the model invented, with paths
  expect_met   - whether the item's `expect` constraints hold (controls and
                 near-miss items use this to check the *right* parameter was
                 used with the right value)
  repurposed   - informational: which of the item's `misuse_params` received a
                 value. These are parameters the request says nothing about, so
                 a value there means the unexpressible constraint was forced
                 into a parameter that means something else (a BCC address in
                 `cc`). Schema-valid, and never caught by a validator.
  enum_choice  - enum_pressure items: what happened to the enum parameter:
                 violated (a value outside the enum), listed (a listed value the
                 user did not ask for), omitted (the parameter was left out)
  stuffed      - informational, never a fault: which of the item's `markers`
                 (words of a constraint the tool cannot express) turned up
                 inside a string argument, i.e. the model packed the
                 constraint into a free-text field instead of dropping it

Primary label order (first match wins):

  unparseable       no single JSON tool call could be recovered
  wrong_tool        the call names a tool other than the target
  phantom_arg       at least one key not defined in the schema (any depth)
  out_of_enum       a value outside an enum (or const)
  wrong_type        a value of the wrong JSON type
  missing_required  a required parameter is absent
  other_invalid     any other schema violation (minimum, maxItems, pattern, ...)
  valid             schema-valid call to the right tool
"""

from dataclasses import dataclass, field
from typing import Any


try:                                    # the real validator when it is installed ...
    from jsonschema import Draft202012Validator as _Validator
    VALIDATOR_BACKEND = "jsonschema"
except ImportError:                     # ... otherwise the built-in subset (same results on this tool library)
    _Validator = MiniValidator
    VALIDATOR_BACKEND = "minischema"

LABEL_ORDER = (
    "unparseable",
    "declined",          # v1.1, may_decline condition only
    "wrong_tool",
    "phantom_arg",
    "out_of_enum",
    "wrong_type",
    "missing_required",
    "other_invalid",
    "repurposed",        # v1.1, items with `misuse` detectors only: schema-valid, wrong parameter
    "valid",
)

_VALIDATOR_TO_FLAG = {
    "additionalProperties": "phantom_arg",
    "unevaluatedProperties": "phantom_arg",
    "enum": "out_of_enum",
    "const": "out_of_enum",
    "type": "wrong_type",
    "required": "missing_required",
}


@dataclass
class Score:
    label: str
    flags: list[str] = field(default_factory=list)
    phantom_keys: list[dict] = field(default_factory=list)   # {"key", "path", "value"}
    enum_violations: list[dict] = field(default_factory=list)  # {"path", "value"}
    type_violations: list[dict] = field(default_factory=list)  # {"path", "expected"}
    missing_required: list[str] = field(default_factory=list)
    other_violations: list[dict] = field(default_factory=list)
    schema_valid: bool = False
    expect_met: bool | None = None       # None when the item has no expectations or the call is not checkable
    expect_failed: list[dict] = field(default_factory=list)
    correct: bool = False                # schema_valid and right tool and expectations met
    stuffed: list[str] = field(default_factory=list)   # item markers found inside string arguments
    repurposed: list[str] = field(default_factory=list)  # parameters that absorbed a constraint they do not mean
    misuse_fired: list[dict] = field(default_factory=list)  # v1.1: the item's `misuse` detectors that matched
    declined: bool = False               # v1.1, may_decline condition: the reply was a decline
    decline_text: str | None = None
    enum_choice: str | None = None       # enum_pressure items: violated | listed | omitted
    enum_value: Any = None               # the value the model put in the item's `enum_param`
    parse_reason: str | None = None
    parse_notes: list[str] = field(default_factory=list)
    called_tool: str | None = None
    arguments: Any = None

    def to_dict(self) -> dict:
        return dict(self.__dict__)


# --------------------------------------------------------------------------
# schema validation
# --------------------------------------------------------------------------
def _path_str(parts) -> str:
    return ".".join(str(p) for p in parts)


def _subschema_properties(schema: dict) -> set[str]:
    return set((schema or {}).get("properties", {}).keys())


def validate_arguments(schema: dict, arguments: dict, validator_cls=None) -> dict:
    """Validate `arguments` against `schema`; return grouped violations."""
    validator = (validator_cls or _Validator)(schema)
    out = {
        "phantom_keys": [],
        "enum_violations": [],
        "type_violations": [],
        "missing_required": [],
        "other_violations": [],
    }
    for err in sorted(validator.iter_errors(arguments), key=lambda e: list(map(str, e.absolute_path))):
        kind = _VALIDATOR_TO_FLAG.get(err.validator)
        path = _path_str(err.absolute_path)
        if kind == "phantom_arg":
            known = _subschema_properties(err.schema)
            extras = [k for k in err.instance.keys() if k not in known] if isinstance(err.instance, dict) else []
            for key in extras:
                out["phantom_keys"].append(
                    {"key": key, "path": path, "value": err.instance[key], "nested": bool(path)}
                )
        elif kind == "out_of_enum":
            out["enum_violations"].append({"path": path, "value": err.instance})
        elif kind == "wrong_type":
            out["type_violations"].append({"path": path, "expected": err.validator_value, "value": err.instance})
        elif kind == "missing_required":
            known_required = err.validator_value
            present = err.instance.keys() if isinstance(err.instance, dict) else ()
            for name in known_required:
                if name not in present:
                    full = f"{path}.{name}" if path else name
                    if full not in out["missing_required"]:
                        out["missing_required"].append(full)
        else:
            out["other_violations"].append({"path": path, "validator": err.validator, "message": err.message[:200]})
    # de-duplicate and order deterministically, whichever validator produced the errors
    seen = set()
    unique = []
    for p in out["phantom_keys"]:
        if (p["path"], p["key"]) not in seen:
            seen.add((p["path"], p["key"]))
            unique.append(p)
    out["phantom_keys"] = sorted(unique, key=lambda p: (p["path"], p["key"]))
    out["missing_required"].sort()
    return out


# --------------------------------------------------------------------------
# expectations
# --------------------------------------------------------------------------
_MISSING = object()


def _get_path(obj: Any, path: str) -> Any:
    cur = obj
    for part in path.split("."):
        if isinstance(cur, dict):
            if part not in cur:
                return _MISSING
            cur = cur[part]
        elif isinstance(cur, list):
            try:
                cur = cur[int(part)]
            except (ValueError, IndexError):
                return _MISSING
        else:
            return _MISSING
    return cur


def _norm(v: Any) -> Any:
    if isinstance(v, str):
        return v.strip().casefold()
    return v


def _eq(a: Any, b: Any) -> bool:
    # bool is not interchangeable with 0/1 here
    if isinstance(a, bool) or isinstance(b, bool):
        return isinstance(a, bool) and isinstance(b, bool) and a == b
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return float(a) == float(b)
    return _norm(a) == _norm(b)


def check_expectation(arguments: dict, exp: dict) -> bool:
    """Evaluate one expectation. Supported ops:

    equals        value at path equals `value` (strings case-insensitive, 5 == 5.0)
    one_of        value at path equals any of `value` (a list)
    contains      string at path contains `value` (case-insensitive);
                  if `value` is a list, any of them
    present       path exists
    absent        path does not exist
    nonempty      path exists and is not null, "", [] or {}
    any_contains  string at path, or any string element of the array at path,
                  contains `value` (case-insensitive); a list means any of them
    set_equals    array at path equals `value` as a set (optionally of `project`ed keys)
    set_contains  array at path contains every element of `value`
    has_item      array at path contains an object whose keys match every
                  key/value in `value` (a dict)
    length        array at path has exactly `value` elements
    """
    op = exp["op"]
    got = _get_path(arguments, exp["path"])
    want = exp.get("value")
    if op == "present":
        return got is not _MISSING
    if op == "absent":
        return got is _MISSING
    if got is _MISSING:
        return False
    if op == "nonempty":
        return got not in (None, "", [], {})
    if op == "any_contains":
        wants = want if isinstance(want, list) else [want]
        hay = [got] if isinstance(got, str) else [g for g in got if isinstance(g, str)] if isinstance(got, list) else []
        return any(_norm(w) in _norm(h) for h in hay for w in wants)
    if op == "equals":
        return _eq(got, want)
    if op == "one_of":
        return any(_eq(got, w) for w in want)
    if op == "contains":
        if not isinstance(got, str):
            return False
        wants = want if isinstance(want, list) else [want]
        return any(_norm(w) in _norm(got) for w in wants)
    if op == "has_item":
        if not isinstance(got, list):
            return False
        return any(
            isinstance(el, dict) and all(k in el and _eq(el[k], v) for k, v in want.items())
            for el in got
        )
    if op == "length":
        return isinstance(got, list) and len(got) == want
    if op in ("set_equals", "set_contains"):
        if not isinstance(got, list):
            return False
        proj = exp.get("project")
        items = []
        for el in got:
            if proj is not None:
                if not isinstance(el, dict) or proj not in el:
                    return False
                el = el[proj]
            items.append(el)
        covered = all(any(_eq(g, w) for g in items) for w in want)
        if op == "set_contains":
            return covered
        return covered and all(any(_eq(g, w) for w in want) for g in items)
    raise ValueError(f"unknown expectation op {op!r}")


def _strings(value: Any):
    """Yield every string value (not key) found anywhere inside `value`."""
    if isinstance(value, str):
        yield value
    elif isinstance(value, dict):
        for v in value.values():
            yield from _strings(v)
    elif isinstance(value, list):
        for v in value:
            yield from _strings(v)


def find_markers(arguments: Any, markers: list[str] | None) -> list[str]:
    """Which markers appear (case-insensitively) inside any string argument."""
    if not markers:
        return []
    haystack = "\n".join(s.casefold() for s in _strings(arguments))
    return [m for m in markers if m.casefold() in haystack]


# --------------------------------------------------------------------------
# scoring
# --------------------------------------------------------------------------
def target_tool(item: dict) -> dict:
    name = item["target_tool"]
    for tool in item["tools"]:
        if tool["name"] == name:
            return tool
    raise KeyError(f"item {item.get('id')}: target tool {name!r} not among its tools")


def score_call(item: dict, parsed: ParsedCall) -> Score:
    if parsed.declined:
        # Only reachable in the may_decline condition. Whether declining is the
        # right answer is a fixed property of the item (`decline_ok`).
        return Score(label="declined", flags=["declined"], declined=True, decline_text=parsed.decline_text,
                     correct=bool(item.get("decline_ok")), parse_reason=parsed.reason,
                     parse_notes=list(parsed.notes))
    if not parsed.ok:
        return Score(
            label="unparseable",
            flags=["unparseable"],
            parse_reason=parsed.reason,
            parse_notes=list(parsed.notes),
            called_tool=parsed.name,
        )

    tool = target_tool(item)
    flags: list[str] = []
    score = Score(label="valid", parse_notes=list(parsed.notes), called_tool=parsed.name, arguments=parsed.arguments)

    wrong_tool = parsed.name != tool["name"]
    if wrong_tool:
        flags.append("wrong_tool")
        # Validate against the tool that was actually called when it is one of
        # the offered tools, so a phantom key on a sibling is still recorded.
        by_name = {t["name"]: t for t in item["tools"]}
        schema_tool = by_name.get(parsed.name)
    else:
        schema_tool = tool

    if schema_tool is not None:
        v = validate_arguments(schema_tool["parameters"], parsed.arguments)
        score.phantom_keys = v["phantom_keys"]
        score.enum_violations = v["enum_violations"]
        score.type_violations = v["type_violations"]
        score.missing_required = v["missing_required"]
        score.other_violations = v["other_violations"]
        if v["phantom_keys"]:
            flags.append("phantom_arg")
        if v["enum_violations"]:
            flags.append("out_of_enum")
        if v["type_violations"]:
            flags.append("wrong_type")
        if v["missing_required"]:
            flags.append("missing_required")
        if v["other_violations"]:
            flags.append("other_invalid")

    score.stuffed = find_markers(parsed.arguments, item.get("markers"))
    if not wrong_tool:
        score.repurposed = [p for p in item.get("misuse_params") or []
                            if parsed.arguments.get(p) not in (None, [], {})]
        enum_param = item.get("enum_param")
        if enum_param:
            if enum_param not in parsed.arguments:
                score.enum_choice = "omitted"
            else:
                score.enum_value = parsed.arguments[enum_param]
                violated = any(v["path"] == enum_param for v in score.enum_violations + score.type_violations)
                score.enum_choice = "violated" if violated else "listed"

    score.schema_valid = not wrong_tool and not any(
        f in flags for f in ("phantom_arg", "out_of_enum", "wrong_type", "missing_required", "other_invalid")
    )

    expectations = item.get("expect") or []
    if expectations and not wrong_tool:
        failed = [e for e in expectations if not check_expectation(parsed.arguments, e)]
        score.expect_failed = failed
        score.expect_met = not failed
    elif expectations:
        score.expect_met = False
    score.correct = score.schema_valid and (score.expect_met is not False)

    # v1.1 `repurposing` items: each detector describes one specific misuse of a
    # parameter that exists but means something else. A match is a fault even
    # though the call is schema-valid. Items without `misuse` are unaffected.
    detectors = item.get("misuse") or []
    if detectors and not wrong_tool:
        fired = [d for d in detectors if check_expectation(parsed.arguments, d)]
        if fired:
            score.misuse_fired = fired
            score.repurposed = sorted({d["path"].split(".")[0] for d in fired} | set(score.repurposed))
            flags.append("repurposed")
            score.correct = False

    score.flags = [f for f in LABEL_ORDER if f in flags]
    score.label = score.flags[0] if score.flags else "valid"
    return score


def score_response(item: dict, text: str | None, condition: str | None = None) -> Score:
    """Parse + score in one step.

    `condition` matters only for "may_decline", where {"cannot_do": "..."} is
    read as a decline. For every other condition (and when it is omitted) the
    reply is parsed and scored exactly as in v1.0.
    """
    parsed = parse_tool_call(text, known_tools=[t["name"] for t in item["tools"]],
                             allow_decline=(condition == "may_decline"))
    return score_call(item, parsed)

In [ ]:
# ----------------------------------------------------------------------------- aggregation and report
# Inlined verbatim from phantom_args/aggregate.py (project repo); covered by the project's tests.
"""Aggregation: per model, per family, per condition, most-invented names.

Works on the raw result records written by run_local.py (one JSON object per
line). Scores are always recomputed from the saved reply text, so a change to
the parser or scorer never requires re-running any model.

Denominators (stated next to every rate):
  scored  - replies that came back complete (not truncated, no API error)
  parsed  - scored replies from which one tool call could be recovered
Phantom and enum rates use `parsed`; control score and parse failures use
`scored` (an unparseable reply to a control item is not a correct call).

CLI:
  python -m phantom_args.aggregate results/pilot_raw.jsonl \
      --items data/items.jsonl --json results/summary.json --md results/pilot_tables.md
"""

import argparse
import json
import math
from collections import Counter, defaultdict
from pathlib import Path
from typing import Iterable


V1_0_FAMILIES = ("conventional", "near_miss", "inexpressible", "enum_pressure", "nested", "control")
FAMILIES = V1_0_FAMILIES + ("repurposing", "matched_control")          # the last two were added in v1.1
SCHEMA_FAULTS = ("wrong_tool", "phantom_arg", "out_of_enum", "wrong_type", "missing_required", "other_invalid")
OUTCOMES = ("repurposed", "declined", "clean_call", "schema_fault", "other_wrong", "unparseable")
PHANTOM_FAMILIES = ("conventional", "near_miss", "inexpressible", "nested")
TEMPTATION_FAMILIES = ("conventional", "near_miss", "inexpressible", "enum_pressure", "nested")


# --------------------------------------------------------------------------
# statistics
# --------------------------------------------------------------------------
def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    """95% Wilson score interval for a proportion."""
    if n == 0:
        return (0.0, 1.0)
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return (max(0.0, centre - half), min(1.0, centre + half))


def mcnemar_exact(b: int, c: int) -> float:
    """Two-sided exact McNemar p-value from the two discordant counts."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return min(1.0, 2 * tail)


def rate(k: int, n: int) -> dict:
    lo, hi = wilson(k, n)
    return {"k": k, "n": n, "rate": (k / n) if n else None, "lo": lo, "hi": hi}


# --------------------------------------------------------------------------
# loading
# --------------------------------------------------------------------------
def load_items(path: str | Path) -> dict[str, dict]:
    items = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if line.strip():
            it = json.loads(line)
            items[it["id"]] = it
    return items


def load_results(paths: Iterable[str | Path]) -> list[dict]:
    """Read result files; the LAST record for a (model, item, condition) wins."""
    latest: dict[tuple, dict] = {}
    for path in paths:
        p = Path(path)
        if not p.exists():
            continue
        with p.open(encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue            # a half-written last line after a crash
                latest[(rec["model"], rec["item_id"], rec["condition"])] = rec
    return list(latest.values())


def rescore(records: list[dict], items: dict[str, dict]) -> list[dict]:
    """Attach a fresh `score` to every complete reply; drop records of unknown items."""
    out = []
    for rec in records:
        item = items.get(rec["item_id"])
        if item is None:
            continue
        if rec.get("prompt_sha") and rec["prompt_sha"] != prompt_sha(item, rec["condition"]):
            continue                    # recorded against an older version of this item
        rec = dict(rec)
        rec["family"] = item["family"]
        rec["core"] = bool(item.get("core"))
        rec["enum_fallback"] = item.get("enum_fallback")
        rec["misuse_item"] = bool(item.get("misuse_params"))
        rec["pattern"] = item.get("pattern")
        if rec.get("status") == "ok":
            rec["score"] = score_response(item, rec.get("content"), rec["condition"]).to_dict()
        else:
            rec["score"] = None
        out.append(rec)
    return out


# --------------------------------------------------------------------------
# cells
# --------------------------------------------------------------------------
def cell(records: list[dict]) -> dict:
    """Counts for one group of records."""
    n_total = len(records)
    n_error = sum(r["status"] == "error" for r in records)
    n_trunc = sum(r["status"] == "truncated" for r in records)
    scored = [r for r in records if r["status"] == "ok"]
    parsed = [r for r in scored if r["score"]["label"] != "unparseable"]
    labels = Counter(r["score"]["label"] for r in scored)
    return {
        "n_total": n_total,
        "n_error": n_error,
        "n_truncated": n_trunc,
        "n_scored": len(scored),
        "n_parsed": len(parsed),
        "labels": dict(labels),
        "unparseable": rate(len(scored) - len(parsed), len(scored)),
        "phantom": rate(sum("phantom_arg" in r["score"]["flags"] for r in parsed), len(parsed)),
        "out_of_enum": rate(sum("out_of_enum" in r["score"]["flags"] for r in parsed), len(parsed)),
        "invalid": rate(sum(r["score"]["label"] != "valid" for r in parsed), len(parsed)),
        "correct": rate(sum(bool(r["score"]["correct"]) for r in scored), len(scored)),
        "stuffed": rate(sum(bool(r["score"]["stuffed"]) and r["score"]["label"] == "valid" for r in parsed), len(parsed)),
        "wrong_tool": rate(sum("wrong_tool" in r["score"]["flags"] for r in parsed), len(parsed)),
        "declined": rate(sum(r["score"]["label"] == "declined" for r in scored), len(scored)),
        "format_deviation": rate(sum(bool(r["score"]["parse_notes"]) for r in parsed), len(parsed)),
    }


def _by(records, *keys):
    groups = defaultdict(list)
    for r in records:
        groups[tuple(r[k] for k in keys)].append(r)
    return groups


def phantom_key_counts(records: list[dict]) -> Counter:
    c: Counter = Counter()
    for r in records:
        if r["status"] == "ok" and r["score"]["phantom_keys"]:
            for key in sorted({p["key"].lower() for p in r["score"]["phantom_keys"]}):   # sorted: stable order for ties
                c[key] += 1
    return c


def paired_effect(records: list[dict], families=PHANTOM_FAMILIES, flag="phantom_arg") -> dict:
    """Neutral vs instructed on the same items (both replies parsed)."""
    by_item: dict[str, dict[str, dict]] = defaultdict(dict)
    for r in records:
        if r["family"] in families and r["status"] == "ok" and r["score"]["label"] != "unparseable":
            by_item[r["item_id"]][r["condition"]] = r
    both = b = c = neither = 0
    for conds in by_item.values():
        if "neutral" not in conds or "instructed" not in conds:
            continue
        n = flag in conds["neutral"]["score"]["flags"]
        i = flag in conds["instructed"]["score"]["flags"]
        if n and i:
            both += 1
        elif n:
            b += 1
        elif i:
            c += 1
        else:
            neither += 1
    return {"pairs": both + b + c + neither, "both": both, "fixed_by_instruction": b,
            "broken_by_instruction": c, "neither": neither, "p_mcnemar": mcnemar_exact(b, c)}


def summarize(records: list[dict]) -> dict:
    models = sorted({r["model"] for r in records})
    summary: dict = {"models": {}, "families": {}, "pooled": {}, "keys": {}, "usage": {}}

    for model in models:
        mrecs = [r for r in records if r["model"] == model]
        entry: dict = {"conditions": {}, "families": {}}
        for cond in CONDITIONS:
            crecs = [r for r in mrecs if r["condition"] == cond]
            if not crecs:
                continue
            entry["conditions"][cond] = {
                "all": cell(crecs),
                "phantom_families": cell([r for r in crecs if r["family"] in PHANTOM_FAMILIES]),
                "enum_pressure": cell([r for r in crecs if r["family"] == "enum_pressure"]),
                "control": cell([r for r in crecs if r["family"] == "control"]),
                "near_miss": cell([r for r in crecs if r["family"] == "near_miss"]),
            }
            for fam in FAMILIES:
                frecs = [r for r in crecs if r["family"] == fam]
                if frecs:
                    entry["families"].setdefault(fam, {})[cond] = cell(frecs)
        entry["paired_phantom"] = paired_effect(mrecs)
        entry["paired_enum"] = paired_effect(mrecs, families=("enum_pressure",), flag="out_of_enum")
        done = [r for r in mrecs if r["status"] in ("ok", "truncated")]
        entry["usage"] = {
            "calls": len(done),
            "prompt_tokens": sum((r.get("usage") or {}).get("prompt_tokens") or 0 for r in done),
            "completion_tokens": sum((r.get("usage") or {}).get("completion_tokens") or 0 for r in done),
            "reasoning_tokens": sum((r.get("usage") or {}).get("reasoning_tokens") or 0 for r in done),
            "calls_with_reasoning": sum(bool((r.get("usage") or {}).get("reasoning_tokens")) or bool(r.get("reasoning_chars")) for r in done),
            "cost": sum(r.get("cost") or 0.0 for r in done),
            "items": len({r["item_id"] for r in mrecs}),
        }
        summary["models"][model] = entry

    for cond in CONDITIONS:
        crecs = [r for r in records if r["condition"] == cond]
        summary["pooled"][cond] = {
            "all": cell(crecs),
            "phantom_families": cell([r for r in crecs if r["family"] in PHANTOM_FAMILIES]),
            "enum_pressure": cell([r for r in crecs if r["family"] == "enum_pressure"]),
            "control": cell([r for r in crecs if r["family"] == "control"]),
        }
        for fam in FAMILIES:
            summary["families"].setdefault(fam, {})[cond] = cell([r for r in crecs if r["family"] == fam])
        temptation = [r for r in crecs if r["family"] in TEMPTATION_FAMILIES]
        counts = phantom_key_counts(temptation)
        models_per_key: dict[str, set] = defaultdict(set)
        for r in temptation:
            if r["status"] == "ok":
                for p in r["score"]["phantom_keys"]:
                    models_per_key[p["key"].lower()].add(r["model"])
        summary["keys"][cond] = [
            {"key": k, "count": n, "models": len(models_per_key[k])} for k, n in counts.most_common(40)
        ]
    summary["paired_phantom_pooled"] = paired_effect_pooled(records)
    summary["reasoning_split"] = reasoning_split(records)
    summary["v1_1"] = v1_1_summary(records)
    summary["unparseable_reasons"] = {
        cond: dict(Counter(r["score"]["parse_reason"] for r in records
                           if r["condition"] == cond and r["status"] == "ok" and r["score"]["label"] == "unparseable"))
        for cond in CONDITIONS}
    summary["unparseable_by_family"] = dict(Counter(
        r["family"] for r in records if r["status"] == "ok" and r["score"]["label"] == "unparseable"))
    summary["item_outcomes"] = item_outcomes(records)
    pooled_enum = Counter()
    for conds in summary["item_outcomes"]["enum"].values():
        for cond, d in conds.items():
            for choice, n in d["choices"].items():
                pooled_enum[(cond, d["fallback"], choice)] += n
    summary["enum_choices_pooled"] = {
        cond: {fb: {c: pooled_enum[(cond, fb, c)] for c in ("violated", "listed", "omitted", "other") if pooled_enum[(cond, fb, c)]}
               for fb in ("omit", "nearest")}
        for cond in CONDITIONS}
    return summary


def _flag_rate(records: list[dict], pred) -> dict:
    parsed = [r for r in records if r["status"] == "ok" and r["score"]["label"] != "unparseable"]
    return rate(sum(bool(pred(r["score"])) for r in parsed), len(parsed))


def reasons_by_default(mrecs: list[dict]) -> bool:
    """True if the provider returned any reasoning (tokens or text) for this model."""
    return any(bool((r.get("usage") or {}).get("reasoning_tokens")) or bool(r.get("reasoning_chars"))
               for r in mrecs if r["status"] in ("ok", "truncated"))


def reasoning_split(records: list[dict]) -> dict:
    """Pool models by whether they reasoned before answering (provider default).

    An association, not an experiment: the two groups are different models, and
    the non-reasoning ones are on the whole older. `core` restricts to the items
    every model answered, so both groups are compared on the same items.
    """
    groups: dict[str, list[str]] = {"reasoning": [], "no_reasoning": []}
    for model in sorted({r["model"] for r in records}):
        mrecs = [r for r in records if r["model"] == model]
        groups["reasoning" if reasons_by_default(mrecs) else "no_reasoning"].append(model)
    out: dict = {"models": groups, "all_items": {}, "core_items": {}}
    for scope, keep in (("all_items", lambda r: True), ("core_items", lambda r: r["core"])):
        for g, members in groups.items():
            for cond in CONDITIONS:
                recs = [r for r in records if r["model"] in members and r["condition"] == cond and keep(r)]
                out[scope].setdefault(g, {})[cond] = {
                    "phantom": cell([r for r in recs if r["family"] in PHANTOM_FAMILIES])["phantom"],
                    "out_of_enum": cell([r for r in recs if r["family"] == "enum_pressure"])["out_of_enum"],
                    "control_correct": cell([r for r in recs if r["family"] == "control"])["correct"],
                    "repurposed": _flag_rate([r for r in recs if r.get("misuse_item")],
                                             lambda sc: bool(sc["repurposed"]) and sc["schema_valid"]),
                    "enum_substituted": _flag_rate([r for r in recs if r.get("enum_fallback") == "omit"],
                                                   lambda sc: sc.get("enum_choice") == "listed"),
                }
    return out


def item_outcomes(records: list[dict]) -> dict:
    """Per-item counts of where an unexpressible request went, over models.

    repurposed  - items with `misuse_params`: schema-valid calls that put a
                  value in a parameter the request gives no value for
    stuffed     - inexpressible items: schema-valid calls with the constraint's
                  words inside a string argument
    enum        - enum_pressure items: violated / listed / omitted, and the
                  values chosen
    """
    out: dict = {"repurposed": {}, "stuffed": {}, "enum": {}}
    for (iid, cond), recs in sorted(_by(records, "item_id", "condition").items()):
        parsed = [r for r in recs if r["status"] == "ok" and r["score"]["label"] != "unparseable"]
        if not parsed:
            continue
        fam = parsed[0]["family"]
        if fam != "repurposing" and any(r["score"].get("repurposed") for r in parsed):
            k = sum(bool(r["score"]["repurposed"]) and r["score"]["schema_valid"] for r in parsed)
            out["repurposed"].setdefault(iid, {})[cond] = {
                **rate(k, len(parsed)),
                "params": dict(Counter(p for r in parsed for p in r["score"]["repurposed"])),
                "phantom": sum("phantom_arg" in r["score"]["flags"] for r in parsed),
            }
        if fam == "inexpressible":
            k = sum(bool(r["score"]["stuffed"]) and r["score"]["schema_valid"] for r in parsed)
            out["stuffed"].setdefault(iid, {})[cond] = {
                **rate(k, len(parsed)), "phantom": sum("phantom_arg" in r["score"]["flags"] for r in parsed)}
        if fam == "enum_pressure":
            choices = Counter(r["score"].get("enum_choice") or "other" for r in parsed)
            values = Counter(json.dumps(r["score"].get("enum_value"), ensure_ascii=False)
                             for r in parsed if r["score"].get("enum_choice") in ("violated", "listed"))
            out["enum"].setdefault(iid, {})[cond] = {
                "n": len(parsed), "fallback": parsed[0].get("enum_fallback"),
                "choices": dict(choices), "values": dict(values.most_common())}
    return out


def outcome(rec: dict) -> str:
    """One of OUTCOMES for a complete reply. Used for the v1.1 families."""
    sc = rec["score"]
    if sc["label"] in ("unparseable", "declined", "repurposed"):
        return sc["label"]       # repurposed = schema-valid; a misuse that also breaks the schema is a schema fault
    if sc["label"] in SCHEMA_FAULTS:
        return "schema_fault"
    return "clean_call" if sc["correct"] else "other_wrong"


def outcome_counts(records: list[dict]) -> dict:
    scored = [r for r in records if r["status"] == "ok"]
    c = Counter(outcome(r) for r in scored)
    return {"n": len(scored), **{o: c.get(o, 0) for o in OUTCOMES},
            "correct": sum(bool(r["score"]["correct"]) for r in scored),
            # a detector matched, but the call is also schema-invalid (e.g. owner_email: null): a strict tool
            # rejects it, so it is a schema fault, counted here so that it is not lost
            "schema_fault_with_misuse": sum(r["score"]["label"] in SCHEMA_FAULTS and bool(r["score"].get("misuse_fired"))
                                            for r in scored)}


def v1_1_summary(records: list[dict]) -> dict:
    """Everything about the repurposing family, the matched controls and the may_decline condition."""
    rep = [r for r in records if r["family"] == "repurposing"]
    ctl = [r for r in records if r["family"] == "matched_control"]
    out: dict = {"models": {}, "pooled": {}, "patterns": {}, "items": {}, "matched_control": {},
                 "decline_by_family": {}, "switch": {}, "reasoning": {}}
    if not rep and not any(r["condition"] == "may_decline" for r in records):
        return out
    conds = [c for c in CONDITIONS if any(r["condition"] == c for r in rep + ctl)]
    out["conditions"] = conds
    models = sorted({r["model"] for r in rep + ctl})
    reasoning = {m: reasons_by_default([r for r in records if r["model"] == m]) for m in models}
    for c in conds:
        out["pooled"][c] = outcome_counts([r for r in rep if r["condition"] == c])
        for g, flag in (("reasoning", True), ("no_reasoning", False)):
            out["reasoning"].setdefault(g, {})[c] = outcome_counts(
                [r for r in rep if r["condition"] == c and reasoning[r["model"]] is flag])
        for pat in sorted({r["pattern"] for r in rep if r["pattern"]}):
            out["patterns"].setdefault(pat, {})[c] = outcome_counts(
                [r for r in rep if r["condition"] == c and r["pattern"] == pat])
        for iid in sorted({r["item_id"] for r in rep}):
            out["items"].setdefault(iid, {})[c] = outcome_counts(
                [r for r in rep if r["condition"] == c and r["item_id"] == iid])
    for m in models:
        out["models"][m] = {"reasoning": reasoning[m],
                            "repurposing": {c: outcome_counts([r for r in rep if r["model"] == m and r["condition"] == c]) for c in conds},
                            "matched_control": {c: outcome_counts([r for r in ctl if r["model"] == m and r["condition"] == c]) for c in conds}}
    for c in conds:
        out["matched_control"][c] = outcome_counts([r for r in ctl if r["condition"] == c])
    # how often a decline is used, per family, when it is allowed
    md = [r for r in records if r["condition"] == "may_decline" and r["status"] == "ok"]
    for fam in FAMILIES:
        fr = [r for r in md if r["family"] == fam]
        if fr:
            out["decline_by_family"][fam] = {**rate(sum(r["score"]["label"] == "declined" for r in fr), len(fr)),
                                             "correct": rate(sum(bool(r["score"]["correct"]) for r in fr), len(fr))}
    # same model, same item: what the must-call outcome turns into when a decline is allowed
    pairs: dict = defaultdict(dict)
    for r in rep:
        if r["status"] == "ok" and r["condition"] in ("neutral", "may_decline"):
            pairs[(r["model"], r["item_id"])][r["condition"]] = outcome(r)
    sw = Counter((p["neutral"], p["may_decline"]) for p in pairs.values() if len(p) == 2)
    out["switch"] = {"pairs": sum(sw.values()), "table": {f"{a} -> {b}": n for (a, b), n in sorted(sw.items(), key=lambda t: -t[1])}}
    b = sum(n for (a, bb), n in sw.items() if a == "repurposed" and bb != "repurposed")
    cc = sum(n for (a, bb), n in sw.items() if a != "repurposed" and bb == "repurposed")
    out["switch"].update({"repurposed_only_when_must_call": b, "repurposed_only_when_may_decline": cc,
                          "p_mcnemar": mcnemar_exact(b, cc)})
    return out


def paired_effect_pooled(records: list[dict]) -> dict:
    tot = Counter()
    for model in sorted({r["model"] for r in records}):
        pe = paired_effect([r for r in records if r["model"] == model])
        for k in ("pairs", "both", "fixed_by_instruction", "broken_by_instruction", "neither"):
            tot[k] += pe[k]
    d = dict(tot)
    d["p_mcnemar"] = mcnemar_exact(tot["fixed_by_instruction"], tot["broken_by_instruction"])
    return d


# --------------------------------------------------------------------------
# markdown
# --------------------------------------------------------------------------
def pct(r: dict, digits: int = 1) -> str:
    if not r["n"]:
        return "n/a"
    return f"{100 * r['rate']:.{digits}f}% ({r['k']}/{r['n']})"


def pct_ci(r: dict) -> str:
    if not r["n"]:
        return "n/a"
    return f"{100 * r['rate']:.1f}% ({r['k']}/{r['n']}; 95% CI {100 * r['lo']:.0f}–{100 * r['hi']:.0f}%)"


def short(model: str) -> str:
    return model.split("/", 1)[-1]


def _oc(d: dict, key: str) -> str:
    return f"{100 * d[key] / d['n']:.1f}% ({d[key]}/{d['n']})" if d and d["n"] else "n/a"


def render_v1_1(v: dict) -> list[str]:
    """Tables for the repurposing family, its matched controls and the may_decline condition."""
    if not v.get("models"):
        return []
    conds = v["conditions"]
    names = {"neutral": "must call", "instructed": "must call + instruction", "may_decline": "may decline"}
    L = ["### v1.1: repurposing, per model\n",
         "Share of complete replies. Repurposed = schema-valid call that puts the request into a parameter that means "
         "something else (the item's detector matched). Clean call = valid call without the misuse that gets the "
         "expressible parts right. Matched controls are requests for which that same parameter is the right one.\n",
         "| Model | Reasoned | Condition | Repurposed | Declined | Clean call | Schema fault | Unparseable | Matched controls correct | Matched controls declined |",
         "|---|---|---|---|---|---|---|---|---|---|"]
    order = sorted(v["models"], key=lambda m: (v["models"][m]["reasoning"], m))
    for m in order:
        e = v["models"][m]
        for c in conds:
            d, k = e["repurposing"].get(c), e["matched_control"].get(c)
            if not d or not d["n"]:
                continue
            L.append(f"| {short(m)} | {'yes' if e['reasoning'] else 'no'} | {names.get(c, c)} | {_oc(d, 'repurposed')} | {_oc(d, 'declined')} | "
                     f"{_oc(d, 'clean_call')} | {_oc(d, 'schema_fault')} | {_oc(d, 'unparseable')} | {_oc(k, 'correct')} | {_oc(k, 'declined')} |")
    L.append("\n### v1.1: repurposing, pooled\n")
    L.append("| Group | Condition | Repurposed | Declined | Clean call | Schema fault | Other wrong | Unparseable |")
    L.append("|---|---|---|---|---|---|---|---|")
    groups = [("all models", v["pooled"])] + [(g.replace("_", " "), v["reasoning"].get(g, {})) for g in ("no_reasoning", "reasoning")]
    groups += [(f"pattern: {p}", d) for p, d in v["patterns"].items()]
    for label, by_cond in groups:
        for c in conds:
            d = by_cond.get(c)
            if d and d["n"]:
                L.append(f"| {label} | {names.get(c, c)} | {_oc(d, 'repurposed')} | {_oc(d, 'declined')} | {_oc(d, 'clean_call')} | "
                         f"{_oc(d, 'schema_fault')} | {_oc(d, 'other_wrong')} | {_oc(d, 'unparseable')} |")
    both = {c: v["pooled"][c].get("schema_fault_with_misuse", 0) for c in conds if v["pooled"].get(c)}
    if any(both.values()):
        L.append("\nSchema faults in which a detector also matched (counted as schema faults, since a strict tool rejects "
                 "them): " + "; ".join(f"{names.get(c, c)}: {n}" for c, n in both.items()) + ".")
    mc = v["matched_control"]
    L.append("\nMatched controls, all models: " + "; ".join(
        f"{names.get(c, c)}: correct {_oc(mc[c], 'correct')}, declined {_oc(mc[c], 'declined')}, flagged as repurposed {_oc(mc[c], 'repurposed')}"
        for c in conds if mc.get(c) and mc[c]["n"]) + ".")
    sw = v.get("switch") or {}
    if sw.get("pairs"):
        L.append(f"\nSame model and item, must call -> may decline ({sw['pairs']} pairs): " +
                 "; ".join(f"{k}: {n}" for k, n in sw["table"].items()) + ".")
        L.append(f"Repurposed only when it had to call: {sw['repurposed_only_when_must_call']}. Repurposed only when it could decline: "
                 f"{sw['repurposed_only_when_may_decline']}. Exact McNemar p "
                 + (f"= {sw['p_mcnemar']:.4f}." if sw['p_mcnemar'] >= 0.0001 else "< 0.0001."))
    if v.get("decline_by_family"):
        L.append("\n### v1.1: how often a decline is used when it is allowed\n")
        L.append("| Family | Declining is | Declined | Correct |")
        L.append("|---|---|---|---|")
        wrong = ("near_miss", "control", "matched_control")
        for fam, d in v["decline_by_family"].items():
            L.append(f"| {fam} | {'wrong' if fam in wrong else 'accepted'} | {pct(d)} | {pct(d['correct'])} |")
    L.append("\n### v1.1: repurposing, per item (models that repurposed / complete replies)\n")
    L.append("| Item | " + " | ".join(names.get(c, c) for c in conds) + " | Declined when allowed |")
    L.append("|---|" + "---|" * (len(conds) + 1))
    for iid, by_cond in v["items"].items():
        cells = [f"{by_cond[c]['repurposed']}/{by_cond[c]['n']}" if by_cond.get(c) and by_cond[c]["n"] else "n/a" for c in conds]
        dd = by_cond.get("may_decline")
        tail = f"{dd['declined']}/{dd['n']}" if dd and dd["n"] else "n/a"
        L.append(f"| {iid} | " + " | ".join(cells) + f" | {tail} |")
    L.append("")
    return L


def render_markdown(summary: dict) -> str:
    L: list[str] = []
    models = summary["models"]

    def cond(m, c, group):
        return models[m]["conditions"].get(c, {}).get(group)

    def neutral_phantom(m):
        g = cond(m, "neutral", "phantom_families")
        return g["phantom"]["rate"] if g and g["phantom"]["n"] else -1.0

    order = sorted(models, key=lambda m: (-neutral_phantom(m), m))

    L.append("### Per model\n")
    L.append("Phantom rate = share of parsed replies with at least one argument name the schema does not define, "
             "over the four phantom families (conventional, near-miss, inexpressible, nested). "
             "Control score = share of control items answered with a schema-valid call that uses the existing parameter correctly. "
             "Counts are k/n.\n")
    L.append("| Model | Items | Phantom, neutral | Phantom, instructed | Paired p | Enum violations, neutral | Enum violations, instructed | Control score, neutral | Control score, instructed | Unparseable (both) | Truncated |")
    L.append("|---|---|---|---|---|---|---|---|---|---|---|")
    for m in order:
        e = models[m]
        row = [short(m), str(e["usage"]["items"])]
        for c in BASE_CONDITIONS:
            g = cond(m, c, "phantom_families")
            row.append(pct(g["phantom"]) if g else "n/a")
        pe = e["paired_phantom"]
        row.append(f"{pe['p_mcnemar']:.3f} ({pe['fixed_by_instruction']} fixed / {pe['broken_by_instruction']} broken)")
        for c in BASE_CONDITIONS:
            g = cond(m, c, "enum_pressure")
            row.append(pct(g["out_of_enum"]) if g else "n/a")
        for c in BASE_CONDITIONS:
            g = cond(m, c, "control")
            row.append(pct(g["correct"]) if g else "n/a")
        k = sum(cond(m, c, "all")["unparseable"]["k"] for c in BASE_CONDITIONS if cond(m, c, "all"))
        n = sum(cond(m, c, "all")["unparseable"]["n"] for c in BASE_CONDITIONS if cond(m, c, "all"))
        row.append(f"{k}/{n}")
        row.append(str(sum(cond(m, c, "all")["n_truncated"] for c in BASE_CONDITIONS if cond(m, c, "all"))))
        L.append("| " + " | ".join(row) + " |")

    ur = summary.get("unparseable_reasons") or {}
    if any(ur.values()):
        names = {"no_json": "answered in prose, no call", "multiple_calls": "two different calls",
                 "malformed_json": "broken JSON", "not_a_tool_call": "JSON that is not a call",
                 "empty": "empty reply", "arguments_not_object": "arguments not an object"}
        L.append("\nUnparseable replies by reason: " + "; ".join(
            f"{c}: " + (", ".join(f"{names.get(k, k)} {v}" for k, v in sorted(ur[c].items(), key=lambda t: -t[1])) or "none")
            for c in CONDITIONS if c in BASE_CONDITIONS or ur.get(c)) + ". By family: " + ", ".join(
            f"{k} {v}" for k, v in sorted(summary["unparseable_by_family"].items(), key=lambda t: -t[1])) + ".")
    L.append("\n### Per family, all models pooled\n")
    L.append("| Family | Target fault | Neutral | Instructed | Any schema violation, neutral | Any schema violation, instructed |")
    L.append("|---|---|---|---|---|---|")
    for fam in V1_0_FAMILIES:
        f = summary["families"][fam]
        if fam == "control":
            L.append(f"| control | (correct call) | {pct_ci(f['neutral']['correct'])} | {pct_ci(f['instructed']['correct'])} | "
                     f"{pct(f['neutral']['invalid'])} | {pct(f['instructed']['invalid'])} |")
            continue
        key = "out_of_enum" if fam == "enum_pressure" else "phantom"
        L.append(f"| {fam} | {key.replace('phantom', 'phantom_arg')} | {pct_ci(f['neutral'][key])} | {pct_ci(f['instructed'][key])} | "
                 f"{pct(f['neutral']['invalid'])} | {pct(f['instructed']['invalid'])} |")

    L.append("\n### Per family and model, neutral → instructed (target-fault rate, k/n)\n")
    fams = [f for f in V1_0_FAMILIES if f != "control"]
    L.append("| Model | " + " | ".join(fams) + " |")
    L.append("|---|" + "---|" * len(fams))
    for m in order:
        row = [short(m)]
        for fam in fams:
            key = "out_of_enum" if fam == "enum_pressure" else "phantom"
            cells = models[m]["families"].get(fam, {})
            parts = []
            for c in BASE_CONDITIONS:
                if c in cells:
                    r = cells[c][key]
                    parts.append(f"{r['k']}/{r['n']}")
                else:
                    parts.append("n/a")
            row.append(" → ".join(parts))
        L.append("| " + " | ".join(row) + " |")

    L.append("\n### Most-invented argument names (temptation items, all models)\n")
    for c in CONDITIONS:
        if c in BASE_CONDITIONS or summary["keys"][c]:
            L.append(f"**{c}**: " + ", ".join(f"`{k['key']}` ×{k['count']} ({k['models']} models)" for k in summary["keys"][c][:25]) + "\n")

    rs = summary.get("reasoning_split")
    if rs and rs["models"]["reasoning"] and rs["models"]["no_reasoning"]:
        L.append("### Models that reasoned before answering vs models that did not\n")
        L.append("Grouped by whether the provider returned any reasoning at its default setting. "
                 "The groups are different models, so this is an association, not the effect of reasoning. "
                 "Core items are the 60 that every model answered.\n")
        L.append(f"- No reasoning ({len(rs['models']['no_reasoning'])}): " + ", ".join(short(m) for m in rs["models"]["no_reasoning"]))
        L.append(f"- Reasoning ({len(rs['models']['reasoning'])}): " + ", ".join(short(m) for m in rs["models"]["reasoning"]) + "\n")
        L.append("| Items | Group | Phantom, neutral | Phantom, instructed | Enum violations, neutral | Enum violations, instructed | Control score, neutral |")
        L.append("|---|---|---|---|---|---|---|")
        for scope, label in (("all_items", "all answered"), ("core_items", "core 60")):
            for g, glabel in (("no_reasoning", "no reasoning"), ("reasoning", "reasoning")):
                d = rs[scope][g]
                L.append(f"| {label} | {glabel} | {pct_ci(d['neutral']['phantom'])} | {pct_ci(d['instructed']['phantom'])} | "
                         f"{pct_ci(d['neutral']['out_of_enum'])} | {pct_ci(d['instructed']['out_of_enum'])} | {pct(d['neutral']['control_correct'])} |")
        L.append("")
        L.append("Schema-valid ways of getting it wrong, same grouping (all answered items). Repurposed = a value in a parameter "
                 "that means something else (items with `misuse_params`). Substituted = on enum items where no listed value "
                 "means what the user asked, a listed value was sent anyway.\n")
        L.append("| Group | Repurposed, neutral | Repurposed, instructed | Enum substituted, neutral | Enum substituted, instructed |")
        L.append("|---|---|---|---|---|")
        for g, glabel in (("no_reasoning", "no reasoning"), ("reasoning", "reasoning")):
            d = rs["all_items"][g]
            L.append(f"| {glabel} | {pct_ci(d['neutral']['repurposed'])} | {pct_ci(d['instructed']['repurposed'])} | "
                     f"{pct_ci(d['neutral']['enum_substituted'])} | {pct_ci(d['instructed']['enum_substituted'])} |")
        L.append("")

    io = summary.get("item_outcomes") or {}
    if io.get("repurposed"):
        L.append("### Schema-valid calls that put the constraint in a parameter meaning something else\n")
        L.append("Items whose tool has a parameter that can absorb the request but means something different. "
                 "k/n = models whose call was schema-valid and gave that parameter a value, over models with a parsed reply.\n")
        L.append("| Item | Parameter | Neutral | Instructed | Phantom instead (neutral) |")
        L.append("|---|---|---|---|---|")
        for iid, conds in io["repurposed"].items():
            params = sorted({p for d in conds.values() for p in d["params"]})
            cells = [pct(conds[c]) if c in conds else "0" for c in CONDITIONS]
            ph = conds.get("neutral", {}).get("phantom", 0)
            L.append(f"| {iid} | `{', '.join(params)}` | {cells[0]} | {cells[1]} | {ph} |")
        L.append("")
    if io.get("stuffed"):
        rows = [(iid, conds) for iid, conds in io["stuffed"].items() if any(d["k"] for d in conds.values())]
        rows.sort(key=lambda t: -(t[1].get("neutral", {}).get("rate") or 0))
        L.append("### Inexpressible items: constraint words written into a string argument (schema-valid)\n")
        L.append("| Item | Neutral | Instructed |")
        L.append("|---|---|---|")
        for iid, conds in rows:
            L.append(f"| {iid} | " + " | ".join(pct(conds[c], 0) if c in conds else "n/a" for c in BASE_CONDITIONS) + " |")
        L.append("")
    if io.get("enum"):
        ep = summary["enum_choices_pooled"]
        L.append("### Enum-pressure items: what happened to the enum parameter\n")
        L.append("violated = a value outside the enum; listed = a listed value the user did not name; omitted = parameter left out. "
                 "Items are split by what a valid call can do: `nearest` (a listed value is a fair stand-in, e.g. high for critical) "
                 "and `omit` (no listed value means what the user asked, so leaving the parameter out is the faithful valid call).\n")
        for c in BASE_CONDITIONS:
            for fb in ("nearest", "omit"):
                d = ep[c].get(fb) or {}
                tot = sum(d.values())
                if tot:
                    L.append(f"- {c}, `{fb}` items: " + ", ".join(f"{k} {v}/{tot} ({100 * v / tot:.1f}%)" for k, v in d.items()))
        L.append("")
        L.append("Neutral condition, values with counts:\n")
        L.append("| Item | Fallback | n | Violated | Listed | Omitted | Values chosen |")
        L.append("|---|---|---|---|---|---|---|")
        for iid, conds in io["enum"].items():
            d = conds.get("neutral")
            if not d:
                continue
            ch = d["choices"]
            vals = ", ".join(f"`{v}` ×{n}" for v, n in d["values"].items())
            L.append(f"| {iid} | {d['fallback']} | {d['n']} | {ch.get('violated', 0)} | {ch.get('listed', 0)} | {ch.get('omitted', 0)} | {vals} |")
        L.append("")

    L.extend(render_v1_1(summary.get("v1_1") or {}))

    L.append("### Usage\n")
    L.append("| Model | Calls | Prompt tokens | Completion tokens | Reasoning tokens reported | Calls with reasoning | Cost (USD) |")
    L.append("|---|---|---|---|---|---|---|")
    tot = Counter()
    for m in sorted(models):
        u = models[m]["usage"]
        L.append(f"| {short(m)} | {u['calls']} | {u['prompt_tokens']:,} | {u['completion_tokens']:,} | {u['reasoning_tokens']:,} | {u['calls_with_reasoning']} | {u['cost']:.4f} |")
        for k in ("calls", "prompt_tokens", "completion_tokens", "reasoning_tokens"):
            tot[k] += u[k]
        tot["cost"] += u["cost"]
    L.append(f"| **total** | {tot['calls']} | {tot['prompt_tokens']:,} | {tot['completion_tokens']:,} | {tot['reasoning_tokens']:,} | | {tot['cost']:.4f} |")
    return "\n".join(L) + "\n"

In [ ]:
# ----------------------------------------------------------------------------- items
# The benchmark data (data/items.jsonl, v1.0, 202 items), zlib + base64. If a dataset containing
# items.jsonl is attached to the notebook it takes precedence, so the data can be updated without touching code.
ITEMS_B64 = """
...
"""


def load_benchmark_items() -> dict:
    attached = sorted(glob.glob("/kaggle/input/**/items.jsonl", recursive=True))   # UNVERIFIED: dataset mount path (CLI docs: /kaggle/input/<dataset-slug>/)
    if attached:
        text = Path(attached[0]).read_text(encoding="utf-8")
        print("items: attached dataset", attached[0])
    else:
        text = zlib.decompress(base64.b64decode("".join(ITEMS_B64.split()))).decode("utf-8")
        print("items: embedded copy")
    out = {}
    for line in text.splitlines():
        if line.strip():
            it = json.loads(line)
            out[it["id"]] = it
    return out


ITEMS = load_benchmark_items()
print(len(ITEMS), "items;", dict(Counter(it["family"] for it in ITEMS.values())))

In [ ]:
# ============================================================================= KAGGLE ADAPTER
# Every call into `kaggle_benchmarks` is in this cell. The one exception is the main task's decorator further
# down, a literal `@kbench.task(...)`: the editor's Benchmark Task panel reads the source and cannot see a wrapper.
# Checked on Kaggle on 2026-10-02 (kaggle-benchmarks 0.6.1, model google/gemini-3.7-flash): every call below
# works as written; the two changes that run showed were needed are marked CHANGED.
#
#   kb_default_llm()                      the model this task is being run for
#   kb_get_llm(slug)                      another model by slug
#   kb_model_name(llm)                    a stable string used as the cache key
#   kb_prompt(llm, text, chat_name, t, n) ONE isolated request, output capped at n tokens -> (reply, usage dict)
#   kb_task(name, description, store)     the task decorator
#   kb_run(task, llm)                     run a task once
#   kb_map(task, llm, rows, n_jobs)       run a task over rows in parallel (may raise; the caller falls back)
#   kb_assert_true / kb_assert_fail       assertions shown in the task result
#   kb_choose(task_name)                  mark the task this notebook publishes
import kaggle_benchmarks as kbench


def kb_default_llm():
    """ASSUMED (quick_start.md): `kbench.llm` is the model the task runs for (LLM_DEFAULT)."""
    return kbench.llm


def kb_get_llm(slug: str):
    """ASSUMED (cookbook.md, "Multi-Model Comparison"): `kbench.llms["vendor/model"]`."""
    return kbench.llms[slug]


def kb_model_name(llm) -> str:
    """ASSUMED: the attribute that holds a model's slug is not documented; try the likely ones."""
    for attr in ("name", "model", "model_name", "slug", "id"):
        value = getattr(llm, attr, None)
        if isinstance(value, str) and value:
            return value
    return os.environ.get("LLM_DEFAULT") or str(llm)


def _kb_usage(chat) -> dict:
    """ASSUMED (user_guide.md, "Token Usage and Cost Tracking"; examples/usage_tracking.py):
    chat.usage.input_tokens / output_tokens / total_cost_nanodollars, None unless the Kaggle Model Proxy is used."""
    usage = getattr(chat, "usage", None)
    get = lambda name: getattr(usage, name, None) if usage is not None else None
    nano = get("total_cost_nanodollars")
    if nano is None:
        parts = [get("input_tokens_cost_nanodollars"), get("output_tokens_cost_nanodollars")]
        nano = sum(p for p in parts if p is not None) if any(p is not None for p in parts) else None
    return {"prompt_tokens": get("input_tokens"), "completion_tokens": get("output_tokens"),
            "cost": (nano / 1e9) if nano is not None else None}


def kb_prompt(llm, text: str, chat_name: str, temperature=0, max_tokens=None) -> tuple:
    """One isolated request. Returns (reply text, {"prompt_tokens", "completion_tokens", "cost"}).

    Verified on Kaggle: `with kbench.chats.new(name) as chat:` opens a fresh conversation; `llm.prompt(text,
    temperature=...)` returns the reply as a string.
    CHANGED (2026-10-03): the output cap goes in as `extra_api_params={"max_tokens": N}`, which reaches the OpenAI
    client's chat.completions.create. Checked on gemini-3.7-flash, claude-haiku-4-5 and gpt-5.4-nano: `max_tokens`
    and `max_completion_tokens` both cap the reply; `max_output_tokens` raises TypeError; inside `extra_body` it is
    ignored. An older library without `extra_api_params` falls back to a call without the cap.
    NOT AVAILABLE: the finish reason. ask() marks a reply truncated when it used the whole cap.
    """
    def send():
        if max_tokens:
            try:
                return llm.prompt(text, temperature=temperature, extra_api_params={"max_tokens": max_tokens})
            except TypeError as exc:
                if "extra_api_params" not in str(exc):
                    raise
        try:
            return llm.prompt(text, temperature=temperature)
        except TypeError:
            return llm.prompt(text)

    try:
        chat_cm = kbench.chats.new(chat_name)
    except AttributeError:                      # no chats.new: send without isolation rather than fail
        reply, usage = send(), {"prompt_tokens": None, "completion_tokens": None, "cost": None}
    else:
        with chat_cm as chat:
            reply = send()
            usage = _kb_usage(chat)
    return (reply if isinstance(reply, str) else str(reply)), usage


def kb_task(name: str, description: str | None = None, store: bool = True):
    """Verified: kbench.task(name, *, description, version, store_task, store_run) used as a decorator.
    `store_task=False` (examples/dataset_evaluation.py) keeps a helper task off the task list."""
    kwargs = {"name": name}
    if description:
        kwargs["description"] = description
    if not store:
        kwargs["store_task"] = False
        kwargs["store_run"] = False         # CHANGED: no run files for the helper (60-200 x 3 files otherwise)
    try:
        return kbench.task(**kwargs)
    except TypeError:
        kwargs.pop("store_task", None)
        kwargs.pop("store_run", None)
        return kbench.task(**kwargs)


def kb_run(task, llm):
    """ASSUMED (quick_start.md): `task.run(llm=...)` returns a run whose `.result` is the task's return value."""
    return task.run(llm=llm)


def kb_map(task, llm, rows: list, n_jobs: int) -> None:
    """Run `task(llm, **row)` for every row, in parallel.
    Verified: workers are threads in this process (joblib ThreadingBackend), so ask() fills RECORDS directly.
    CHANGED: no `kbench.client.enable_cache()`. Its cache key is task name + row index + model, not the row, so
    a second evaluate in one session (probe then full set, another condition) got the first one's results
    back without running. ask() already caches by (model, item, condition). Any exception here makes the
    caller fall back to a plain sequential loop."""
    import pandas as pd
    task.evaluate(llm=[llm], evaluation_data=pd.DataFrame(rows), on_failure="continue", max_attempts=2,
                  retry_delay=5, n_jobs=n_jobs, timeout=300)


def kb_assert_true(condition: bool, expectation: str) -> None:
    """ASSUMED (user_guide.md): `kbench.assertions.assert_true(value, expectation=...)`."""
    try:
        kbench.assertions.assert_true(condition, expectation=expectation)
    except AttributeError:
        print("assertion", "passed" if condition else "FAILED", "-", expectation)


def kb_assert_fail(expectation: str) -> None:
    try:
        kbench.assertions.assert_fail(expectation=expectation)
    except AttributeError:
        print("assertion FAILED -", expectation)


def kb_choose(task_name: str) -> None:
    """ASSUMED (quick_start.md): the leaderboard takes ONE task per notebook, selected with the cell magic
    `%choose <task name>`. In the .ipynb this cell is the literal magic; here it is invoked from code."""
    try:
        get_ipython().run_line_magic("choose", task_name)
    except NameError:
        pass                                    # not running inside IPython
# ============================================================================= END OF KAGGLE ADAPTER

In [ ]:
# ----------------------------------------------------------------------------- cache, budget, one call
FAMILY_ORDER = FAMILIES_ALL = ("conventional", "near_miss", "inexpressible", "enum_pressure", "nested", "control",
                               "repurposing", "matched_control")
if CONDITION not in CONDITIONS:
    raise ValueError(f"CONDITION must be one of {CONDITIONS}")
if ONLY_FAMILIES:
    unknown = set(ONLY_FAMILIES) - set(FAMILIES_ALL)
    if unknown:
        raise ValueError(f"unknown families {sorted(unknown)}")
    ITEMS = {iid: it for iid, it in ITEMS.items() if it["family"] in ONLY_FAMILIES}
_LOCK = threading.Lock()
CACHE_FILE = Path(CACHE_DIR) / "phantom_records.jsonl"
RECORDS: dict = {}            # (model, item_id, condition) -> record, same schema as the local runner's results file


class BudgetStop(RuntimeError):
    pass


def _load_cache() -> None:
    """Read every record file we can find: this session's, and any attached as a dataset from an earlier
    session (attach the previous notebook version's output to resume across days without paying again)."""
    paths = [str(CACHE_FILE)] + sorted(glob.glob("/kaggle/input/**/phantom_records*.jsonl", recursive=True))
    for path in paths:
        if not os.path.exists(path):
            continue
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                item = ITEMS.get(rec.get("item_id"))
                if item is None or rec.get("status") != "ok" or rec.get("condition") not in CONDITIONS:
                    continue
                if rec.get("prompt_sha") != prompt_sha(item, rec["condition"]):
                    continue                                   # recorded for an older version of the item
                RECORDS[(rec["model"], rec["item_id"], rec["condition"])] = rec
    print(f"cache: {len(RECORDS)} replies already on disk")


SESSION = {"spent_usd": 0.0, "paid_calls": 0, "calls_without_cost": 0}


def ask(llm, item: dict, condition: str) -> dict:
    """One benchmark call, cached. Never repeats a call whose reply is already on disk."""
    model = kb_model_name(llm)
    key = (model, item["id"], condition)
    with _LOCK:
        if key in RECORDS:
            return RECORDS[key]
        if SESSION["spent_usd"] >= SESSION_BUDGET_USD:
            raise BudgetStop(f"session budget of ${SESSION_BUDGET_USD:.2f} reached (${SESSION['spent_usd']:.2f} spent)")
        if SESSION["calls_without_cost"] >= MAX_CALLS_IF_COST_UNKNOWN:
            raise BudgetStop(f"{MAX_CALLS_IF_COST_UNKNOWN} calls made without any cost information; stopping to be safe")
    t0 = time.time()
    reply, usage = kb_prompt(llm, build_prompt(item, condition), f"{item['id']}:{condition}", TEMPERATURE,
                             MAX_OUTPUT_TOKENS)
    rec = {"ts": time.time(), "model": model, "item_id": item["id"], "family": item["family"], "condition": condition,
           "prompt_sha": prompt_sha(item, condition), "status": "ok", "content": reply, "usage": usage,
           "cost": usage["cost"] or 0.0, "latency_s": round(time.time() - t0, 2), "source": "kaggle",
           "max_tokens": MAX_OUTPUT_TOKENS,
           "truncated": bool(MAX_OUTPUT_TOKENS and (usage["completion_tokens"] or 0) >= MAX_OUTPUT_TOKENS)}
    with _LOCK:
        RECORDS[key] = rec
        SESSION["paid_calls"] += 1
        if usage["cost"] is None:
            SESSION["calls_without_cost"] += 1
        else:
            SESSION["spent_usd"] += usage["cost"]
        with open(CACHE_FILE, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
    return rec


def print_spend(prefix: str = "") -> None:
    note = "" if not SESSION["calls_without_cost"] else f" ({SESSION['calls_without_cost']} calls reported no cost)"
    print(f"{prefix}session spend so far: ${SESSION['spent_usd']:.4f} over {SESSION['paid_calls']} paid calls{note}; "
          f"session budget ${SESSION_BUDGET_USD:.2f}. Account-level figure: run `kaggle benchmarks quota` or see the notebook's quota panel.")


_load_cache()

In [ ]:
# ----------------------------------------------------------------------------- cost estimate BEFORE running
# Token counts per call measured in the local pilot (24 open-weight models, Nebius, 2026-10-02):
#   prompt: about 330 tokens.  completion: about 45 tokens without reasoning, 140-690 with reasoning on by default.
# Fill PRICES with Kaggle's per-million-token prices for the models you plan to run to get a table up front;
# the task also measures the real cost on a few probe items before it commits to the full item set.
PRICES_USD_PER_MTOK = {
    # "vendor/model-slug": (input_price, output_price),
}
PROMPT_TOKENS_PER_CALL = 330
COMPLETION_TOKENS_PER_CALL = {"no reasoning": 45, "light reasoning": 200, "heavy reasoning": 600}
N_FULL = len(ITEMS)
N_CORE = sum(bool(it.get("core")) for it in ITEMS.values())


def estimate_table() -> None:
    per_family = dict(Counter(it["family"] for it in ITEMS.values()))
    print(f"calls per model and condition: full {N_FULL} {per_family}, core {N_CORE} (at most 10 per family)")
    for slug, (p_in, p_out) in PRICES_USD_PER_MTOK.items():
        for label, out_tok in COMPLETION_TOKENS_PER_CALL.items():
            per_call = (PROMPT_TOKENS_PER_CALL * p_in + out_tok * p_out) / 1e6
            print(f"  {slug:40s} {label:16s} full ${per_call * N_FULL:.2f}  core ${per_call * N_CORE:.2f}   (one condition)")
    if not PRICES_USD_PER_MTOK:
        print("  PRICES_USD_PER_MTOK is empty: no up-front table; the probe inside the task decides full vs core.")


estimate_table()

In [ ]:
# ----------------------------------------------------------------------------- the tasks
def choose_items(model: str, condition: str) -> list:
    """Items to run for this model: probe first, then the full set or the core subset."""
    everything = sorted(ITEMS.values(), key=lambda it: it["id"])
    core = [it for it in everything if it.get("core")]
    if SUBSET == "full":
        return everything
    if SUBSET == "core":
        return core
    # auto: project the cost of the full set from what this model's replies have cost so far
    seen = [r for (m, _, c), r in RECORDS.items() if m == model and r.get("cost")]
    if not seen:
        return core                       # no cost information yet (or the proxy reports none): stay small
    mean = sum(r["cost"] for r in seen) / len(seen)
    projected = mean * len(everything)
    choice = everything if projected <= PER_MODEL_BUDGET_USD else core
    print(f"{model}: ${mean * 1000:.3f} per 1k calls -> full set projected at ${projected:.2f} per condition; "
          f"running {'FULL' if choice is everything else 'CORE'} ({len(choice)} items)")
    return choice


@kb_task("phantom_args_item", store=False)
def phantom_item(llm, item_id: str, condition: str) -> dict:
    """One item: prompt, parse, validate against the tool's JSON Schema."""
    rec = ask(llm, ITEMS[item_id], condition)
    score = score_response(ITEMS[item_id], rec["content"], condition)
    return {"item_id": item_id, "label": score.label, "correct": score.correct}


def run_items(llm, items: list, condition: str) -> None:
    """Fill the cache for these items. Parallel through the library when possible, else a plain loop."""
    model = kb_model_name(llm)
    todo = [it for it in items if (model, it["id"], condition) not in RECORDS]
    if not todo:
        return
    if PARALLEL_JOBS > 0:
        try:
            kb_map(phantom_item, llm, [{"item_id": it["id"], "condition": condition} for it in todo], PARALLEL_JOBS)
        except BudgetStop:
            raise
        except Exception as exc:                                  # API mismatch: fall back to the plain loop
            print(f"parallel run failed ({type(exc).__name__}: {str(exc)[:200]}); falling back to a sequential loop")
        missing = [it for it in todo if (model, it["id"], condition) not in RECORDS]
        if missing and SESSION["spent_usd"] < SESSION_BUDGET_USD:
            print(f"{len(missing)} items still unanswered after the parallel pass; retrying them one at a time")
    for it in todo:                                               # sequential pass: only items still missing
        if (model, it["id"], condition) in RECORDS:
            continue
        try:
            ask(llm, it, condition)
        except BudgetStop as stop:
            print("STOPPED:", stop)
            break
        except Exception as exc:
            print(f"{it['id']}: call failed ({type(exc).__name__}: {str(exc)[:160]}); skipped, will be retried on the next run")


def probe_items() -> list:
    """One core item per family: a few calls that measure the real cost per call for this model."""
    out = []
    for fam in FAMILY_ORDER:
        fam_items = sorted((it for it in ITEMS.values() if it["family"] == fam and it.get("core")), key=lambda it: it["id"])
        out.extend(fam_items[:1])
    return out


def scored_records(model: str, condition: str | None = None) -> list:
    recs = [r for (m, _, c), r in RECORDS.items() if m == model and (condition is None or c == condition)]
    return rescore(recs, ITEMS)


# The main task uses the library decorator itself with a literal name, not kb_task or TASK_NAME: the editor's
# Benchmark Task panel and `kaggle b t push` find the task by reading the source, and see neither a wrapper nor a
# variable. Keep this literal equal to TASK_NAME (the project's tests check it in every generated notebook).
@kbench.task(name="phantom_args_instructed",
             description="Share of requests answered correctly when the tool cannot do what was asked: a schema-valid call "
         "that misuses no parameter (in the may_decline condition, a decline where declining is right). JSON Schema "
         "checks only, no LLM judge.")   # Kaggle rejects a task name or description over 255 characters
def phantom_args(llm) -> float:
    """Leaderboard score = share of items answered correctly. Higher is better.

    correct = a schema-valid call to the right tool that meets the item's checks and, on `repurposing` items,
    does not put the request into a parameter that means something else. In the `may_decline` condition a
    decline counts as correct on items the tool cannot serve and as wrong on controls."""
    model = kb_model_name(llm)
    if SUBSET == "auto":
        run_items(llm, probe_items(), CONDITION)
    run_items(llm, choose_items(model, CONDITION), CONDITION)
    recs = scored_records(model, CONDITION)
    print_spend(f"[{model}] ")
    if not recs:
        kb_assert_fail("At least one item should have been answered.")
        return 0.0
    correct = sum(bool(r["score"]["correct"]) for r in recs)
    print(render_markdown(summarize(scored_records(model))))
    kb_assert_true(len(recs) >= len({it["family"] for it in ITEMS.values()}),
                   "The run should cover at least one item per family.")
    return correct / len(recs)

In [ ]:
# ----------------------------------------------------------------------------- run
run = kb_run(phantom_args, kb_default_llm())
run

In [ ]:
# Optional: more models in this interactive session (does not create leaderboard entries by itself;
# leaderboard entries come from running the saved task against each model).
for slug in EXTRA_MODELS:
    try:
        kb_run(phantom_args, kb_get_llm(slug))
    except BudgetStop as stop:
        print("STOPPED:", stop)
        break
    except Exception as exc:
        print(f"{slug}: {type(exc).__name__}: {str(exc)[:200]}")

In [ ]:
# ----------------------------------------------------------------------------- report over everything cached
all_records = rescore(list(RECORDS.values()), ITEMS)
if all_records:
    print(render_markdown(summarize(all_records)))
print_spend()
print(f"raw replies: {CACHE_FILE} (same format as the local runner; `python -m phantom_args.aggregate` reads it)")

In [ ]:
# The leaderboard takes ONE task per notebook (quick_start.md, "%choose").
# The name must match CONDITION: phantom_args_neutral, phantom_args_instructed or phantom_args_may_decline.
%choose phantom_args_instructed